# Lab 01 & 02 — الأساسيات، إدارة السياق، التوجيه والاعتمادية

**نظرة عامة:** بناء الأساس لمساعد Farah Hospital AI من خلال واجهة موحدة للـLLM، ذاكرة محادثة قصيرة، توجيه الطلبات، أكثر من AI Client، قياس زمن الاستجابة، Fallback، واكتشاف الحالات الطارئة.

راجع ملف `README.md` داخل هذا المجلد للحصول على شرح مختصر ومنظم للأهداف، طريقة العمل، النتائج، وأهم النقاط التعليمية.


In [1]:
def ask_farah_hospital (question):
  return f"Farah Hospital received: {question}"

In [2]:
print(
    ask_farah_hospital(
        "How do I book an appointment"
    )
)

Farah Hospital received: How do I book an appointment


In [3]:
from dataclasses import dataclass, field
from typing import Protocol

@dataclass
class LLMRequest:
  messages: list[dict]
  model: str
  max_tokens: int = 512

@dataclass
class LLMResponse:
  text: str

class LLMClient(Protocol):
  def complete(self, request: LLMRequest) -> LLMResponse:
    ...

In [4]:
class FarahHospitalClient:
  def complete(self,request: LLMRequest) -> LLMResponse:
    question = request.messages[-1]["content"]
    return LLMResponse(
        text=f"Farah Hospital response to: {question}"
    )

In [5]:
farah_hospital = FarahHospitalClient()
request = LLMRequest(
    messages=[
        {
        "role": "user",
        "content": "How do I book an appointment?"
        }
    ],
    model="demo-model"
)
response = farah_hospital.complete(request)
print(response.text)


Farah Hospital response to: How do I book an appointment?


In [6]:
request = LLMRequest(
    messages=[
        {
            "role": "user",
            "content": "Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟"
            }
    ],
    model="demo-model"
)
response = farah_hospital.complete(request)
print(response.text)

Farah Hospital response to: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟


In [7]:
@dataclass
class ConversationState:
    max_turns: int = 8
    messages: list[dict] = field(default_factory=list)

    def add(self, role, content):
        self.messages.append({
            "role": role,
            "content": content
        })

        self.messages = self.messages[
            -(self.max_turns * 2):
        ]

    def get_messages(self):
        return self.messages

In [8]:
conversation = ConversationState(max_turns=8)

conversation.add(
    "user",
    "My name is Farah and I need a medical appointment."
)

conversation.add(
    "assistant",
    "Welcome Farah. I can help you find the right department."
)

conversation.add(
    "user",
    "I have been experiencing severe headaches."
)

conversation.add(
    "assistant",
    "I can guide you to the appropriate hospital department."
)

print(conversation.get_messages())

[{'role': 'user', 'content': 'My name is Farah and I need a medical appointment.'}, {'role': 'assistant', 'content': 'Welcome Farah. I can help you find the right department.'}, {'role': 'user', 'content': 'I have been experiencing severe headaches.'}, {'role': 'assistant', 'content': 'I can guide you to the appropriate hospital department.'}]


In [9]:
def make_chat(conversation, client, user_message):
    conversation.add(
        "user",
        user_message
    )

    request = LLMRequest(
        messages=conversation.get_messages(),
        model="farah-hospital-ai"
    )

    response = client.complete(request)

    conversation.add(
        "assistant",
        response.text
    )

    return response.text

In [10]:
conversation = ConversationState(max_turns=8)
farah_hospital = FarahHospitalClient()

In [11]:
print(
    make_chat(
        conversation,
        farah_hospital,
        "I have severe headaches and dizziness. Which hospital department should I visit?"
    )
)

Farah Hospital response to: I have severe headaches and dizziness. Which hospital department should I visit?


In [12]:
print(
    make_chat(
        conversation,
        farah_hospital,
        "Can you remember my symptoms and help me book the right specialist?"
    )
)

Farah Hospital response to: Can you remember my symptoms and help me book the right specialist?


In [13]:
conversation = ConversationState(max_turns=8)

for i in range(1, 10):
    conversation.add(
        "user",
        f"Patient message {i}"
    )

    conversation.add(
        "assistant",
        f"Farah Hospital response {i}"
    )

print(conversation.get_messages())

[{'role': 'user', 'content': 'Patient message 2'}, {'role': 'assistant', 'content': 'Farah Hospital response 2'}, {'role': 'user', 'content': 'Patient message 3'}, {'role': 'assistant', 'content': 'Farah Hospital response 3'}, {'role': 'user', 'content': 'Patient message 4'}, {'role': 'assistant', 'content': 'Farah Hospital response 4'}, {'role': 'user', 'content': 'Patient message 5'}, {'role': 'assistant', 'content': 'Farah Hospital response 5'}, {'role': 'user', 'content': 'Patient message 6'}, {'role': 'assistant', 'content': 'Farah Hospital response 6'}, {'role': 'user', 'content': 'Patient message 7'}, {'role': 'assistant', 'content': 'Farah Hospital response 7'}, {'role': 'user', 'content': 'Patient message 8'}, {'role': 'assistant', 'content': 'Farah Hospital response 8'}, {'role': 'user', 'content': 'Patient message 9'}, {'role': 'assistant', 'content': 'Farah Hospital response 9'}]


In [14]:
from pathlib import Path

Path("docs").mkdir(exist_ok=True)

Path("docs/context_budget.md").write_text(
"""# Farah Hospital AI Assistant - Context Budget

Total Context Budget: 16K tokens

- System Prompt: 2K
- Conversation History: 8K
- Hospital Tools & Services: 2K
- AI Response Output: 4K

The conversation history is limited to 8 turns
to maintain fast and efficient patient assistance.

Project: Farah Hospital AI Assistant
Purpose: Smart patient guidance, appointment assistance,
department recommendation, and conversation memory.
""",
encoding="utf-8"
)

print("Farah Hospital AI documentation created successfully!")

Farah Hospital AI documentation created successfully!


In [15]:
def hospital_router(question):

    q = question.lower()

    if any(word in q for word in
           ["heart", "chest", "cardiology", "قلب", "صدر"]):
        return "Cardiology"

    elif any(word in q for word in
             ["skin", "rash", "dermatology", "جلد", "حبوب"]):
        return "Dermatology"

    elif any(word in q for word in
             ["child", "baby", "pediatric", "طفل", "أطفال"]):
        return "Pediatrics"

    elif any(word in q for word in
             ["eye", "vision", "عين", "نظر"]):
        return "Ophthalmology"

    elif any(word in q for word in
             ["appointment", "book", "موعد", "حجز"]):
        return "Appointments"

    else:
        return "General Medicine"

In [16]:
questions = [
    "I have chest pain",
    "I have a skin rash",
    "My child has a fever",
    "I need an eye appointment",
    "أريد حجز موعد"
]

for question in questions:
    department = hospital_router(question)

    print("Patient:", question)
    print("Recommended Department:", department)
    print("-" * 40)

Patient: I have chest pain
Recommended Department: Cardiology
----------------------------------------
Patient: I have a skin rash
Recommended Department: Dermatology
----------------------------------------
Patient: My child has a fever
Recommended Department: Pediatrics
----------------------------------------
Patient: I need an eye appointment
Recommended Department: Ophthalmology
----------------------------------------
Patient: أريد حجز موعد
Recommended Department: Appointments
----------------------------------------


In [17]:
def farah_hospital_assistant(question):

    department = hospital_router(question)

    response = f"""
🏥 FARAH HOSPITAL AI

Patient Request:
{question}

Recommended Department:
{department}

Action:
Your request has been analyzed and routed to the appropriate hospital service.

Status:
Ready for appointment processing.
"""

    return response


print(
    farah_hospital_assistant(
        "I have a skin problem and I need an appointment"
    )
)


🏥 FARAH HOSPITAL AI

Patient Request:
I have a skin problem and I need an appointment

Recommended Department:
Dermatology

Action:
Your request has been analyzed and routed to the appropriate hospital service.

Status:
Ready for appointment processing.



In [18]:
class OpenAICompatClient:
  def __init__(self, name: str):
    self.name = name

  def complete(self, request: LLMRequest) -> LLMResponse:
    question = request.messages[-1]["content"]
    return LLMResponse(text=f"[{self.name}] Processing your request about: {question}")

class AnthropicClient:
  def __init__(self, name: str):
    self.name = name

  def complete(self, request: LLMRequest) -> LLMResponse:
    question = request.messages[-1]["content"]
    return LLMResponse(text=f"[{self.name}] Expert analysis of: {question}")

clients = [
    OpenAICompatClient(name="Farah Fast AI"),
    AnthropicClient(name="Farah Medical Expert")
]

for client in clients:
    response = client.complete(request)
    print(
        client.__class__.__name__,
        "→",
        response.text
    )

OpenAICompatClient → [Farah Fast AI] Processing your request about: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟
AnthropicClient → [Farah Medical Expert] Expert analysis of: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟


In [19]:
routes = {
    "primary": AnthropicClient(
        name="Farah Medical Expert"
    ),

    "fast": OpenAICompatClient(
        name="Farah Fast AI"
    ),

    "vllm": OpenAICompatClient(
        name="Farah Local Medical AI"
    )
}

for route, client in routes.items():

    response = client.complete(request)

    print(
        route,
        "→",
        response.text
    )

primary → [Farah Medical Expert] Expert analysis of: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟
fast → [Farah Fast AI] Processing your request about: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟
vllm → [Farah Local Medical AI] Processing your request about: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟


In [20]:
import time

def stream_chat(client, request):

    response = client.complete(request)
    words = response.text.split()

    start = time.time()
    ttft = None

    time.sleep(0.4)

    for word in words:

        if ttft is None:
            ttft = time.time() - start

        print(
            word,
            end=" ",
            flush=True
        )

        time.sleep(0.1)

    total = time.time() - start

    print()
    print(
        "First response time:",
        round(ttft, 2),
        "seconds"
    )

    print(
        "Total response time:",
        round(total, 2),
        "seconds"
    )


stream_chat(
    routes["primary"],
    request
)

[Farah Medical Expert] Expert analysis of: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟ 
First response time: 0.4 seconds
Total response time: 2.21 seconds


In [21]:
class BrokenMedicalClient:

    def __init__(self):
        self.name = "Primary Medical AI"

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:

        raise Exception(
            "503 Medical AI Service Unavailable"
        )

In [22]:
class ResilientHospitalAI:

    def __init__(
        self,
        primary,
        fallback,
        max_attempts=3
    ):

        self.primary = primary
        self.fallback = fallback
        self.max_attempts = max_attempts


    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:

        for attempt in range(
            1,
            self.max_attempts + 1
        ):

            try:

                print(
                    "Trying Primary AI...",
                    attempt
                )

                return self.primary.complete(
                    request
                )

            except Exception as error:

                print(
                    "Primary failed:",
                    error
                )

                time.sleep(0.3)


        print(
            "⚠️ Switching to backup medical AI..."
        )

        return self.fallback.complete(
            request
        )

In [23]:
hospital_ai = ResilientHospitalAI(

    primary=BrokenMedicalClient(),

    fallback=routes["vllm"]
)

response = hospital_ai.complete(
    request
)

print()
print("Final Response:")
print(response.text)

Trying Primary AI... 1
Primary failed: 503 Medical AI Service Unavailable
Trying Primary AI... 2
Primary failed: 503 Medical AI Service Unavailable
Trying Primary AI... 3
Primary failed: 503 Medical AI Service Unavailable
⚠️ Switching to backup medical AI...

Final Response:
[Farah Local Medical AI] Processing your request about: Farah Hospital لدي صداع مستمر ، اي قسم يجب ان احجز فيه؟


In [24]:
def detect_priority(question):

    q = question.lower()

    emergency_words = [
        "chest pain",
        "can't breathe",
        "unconscious",
        "severe bleeding",
        "difficulty breathing",
        "ألم شديد في الصدر",
        "لا أستطيع التنفس",
        "نزيف شديد",
        "فاقد الوعي"
    ]

    for word in emergency_words:

        if word in q:
            return "EMERGENCY"

    return "NORMAL"

In [25]:
def display_patient_summary(question, department, priority):
    print("------------------------------")
    print("Patient Request:")
    print(question)
    print()
    print("AI Recommended Department:")
    print(department)
    print()
    print("Priority:")
    print(priority)

    if priority == "EMERGENCY":
        print()
        print(
            "🚨 URGENT: Immediate medical evaluation required."
        )
    else:
        print()
        print(
            "✅ Patient can continue to appointment services."
        )

# Example usage:
# display_patient_summary("I have chest pain", "Cardiology", "EMERGENCY")

In [26]:
def smart_triage(question):
    department = hospital_router(question)
    priority = detect_priority(question)
    display_patient_summary(question, department, priority)

smart_triage(
    "I have severe chest pain"
)

------------------------------
Patient Request:
I have severe chest pain

AI Recommended Department:
Cardiology

Priority:
EMERGENCY

🚨 URGENT: Immediate medical evaluation required.


In [27]:
delays = {
    "primary": 1.0,
    "fast": 0.6,
    "vllm": 1.4
}

test_questions = [

    "How do I book an appointment?",

    "Which department treats heart problems?",

    "I need a dermatologist",

    "أريد حجز موعد في عيادة العيون"
]


for route, client in routes.items():

    start = time.time()

    for question in test_questions:

        test_request = LLMRequest(

            messages=[
                {
                    "role": "user",
                    "content": question
                }
            ],

            model="farah-medical-model"
        )

        time.sleep(
            delays[route]
        )

        client.complete(
            test_request
        )


    average = (
        time.time() - start
    ) / len(test_questions)


    print(
        route,
        "| Average:",
        round(average, 2),
        "seconds"
    )

primary | Average: 1.0 seconds
fast | Average: 0.6 seconds
vllm | Average: 1.4 seconds
